# Add Test-Validation Split task to LabelStudio
This task is primarily about having the page information and the test-validation split information in LabelStudio. If needed, the annotator can override the pre-assigned test-validation split and also mark a page to exclude from the evaluation.

In [1]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
from label_studio_sdk.client import LabelStudio

# Load environment variables
load_dotenv()

# Direct the notebook to find scripts in parent directory, since it is sitting one level down
os.chdir("..")

# Import the new task classes
from adt_labelstudio.validation_test_split import ValidationTestSplitTask
from adt_labelstudio.utils import get_project_annotations, get_ls_project_id_from_name


In [2]:

# Connect to the Label Studio API and check the connection
LABEL_STUDIO_URL = "https://" + os.getenv("LABEL_STUDIO_HOST")
API_KEY = os.getenv("LABEL_STUDIO_TOKEN")

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

validation_test_split_task = ValidationTestSplitTask()

## TASK: `Test-Validation Split`

In [3]:
# Get the data used to populate this task 
# (as we are not using pre-existing annotations, we pull the data on books and pages from the adt-gold-standard Github repo)
pages = pd.read_csv('../adt-gold-standard/pages/pages.csv')
books = pd.read_csv("../adt-gold-standard/books/books.csv", usecols=["book_id","book_title"])

# Add book_title to pages
pages = pages.merge(books, on="book_id")

In [4]:
target_project_name = "A0: Validation-Test Split"

# Get annotations that have already been done for this task
target_project_tasks = get_project_annotations(ls_client, project_name=target_project_name)

tasks_to_load = []

for i, page_dict in pages.iterrows():
    book_id, page_id = page_dict['book_id'], page_dict['page_id']
    print(book_id, ", page", page_id)
    
    # Confirm that task does not already exist
    if target_project_tasks.shape[0]>0 and (book_id, page_id) in [xy for xy in zip(target_project_tasks['book_id'], target_project_tasks['page_id'])]:
        print(f"Task for {book_id} page {page_id} already exists in LabelStudio and was not added.")#
        continue

    # Create labelstudio task, consisting of input data and predictions
    task_data = validation_test_split_task.populate_task_data(page_dict)
    task_predictions = validation_test_split_task.populate_task_predictions(page_dict)
    task_json = validation_test_split_task.create_one_task(task_data, task_predictions)

    tasks_to_load.append(task_json)

# Add the whole list to LabelStudio
if len(tasks_to_load) > 0:
    target_project_id  = get_ls_project_id_from_name(ls_client, project_name=target_project_name)

    ls_client.projects.import_tasks(
                id=target_project_id,
                request=tasks_to_load
        )

with open('adt_labelstudio/tasks_to_upload/A0_validation_test_split_tasks.json', 'w') as f:
        json.dump(tasks_to_load, f)

LP-10 , page 5
LP-11 , page 3
LP-11 , page 4
LP-12 , page 1
LP-12 , page 8
LP-12 , page 3
LP-12 , page 4
LP-13 , page 1
LP-13 , page 5
LP-13 , page 4
LP-15 , page 2
LP-15 , page 4
LP-16 , page 1
LP-16 , page 6
LP-16 , page 8
LP-18 , page 1
LP-18 , page 6
LP-18 , page 3
LP-18 , page 4
LP-20 , page 17
LP-22 , page 4
LP-22 , page 5
LP-22 , page 8
LP-22 , page 12
LP-23 , page 11
LP-4 , page 1
LP-4 , page 2
LP-4 , page 4
LP-4 , page 26
LP-4 , page 42
LP-9 , page 1
C-98 , page 15
C-1 , page 10
W-38 , page 1
W-38 , page 54
W-38 , page 61
W-38 , page 72
W-38 , page 76
W-38 , page 77
W-38 , page 81
C-49 , page 52
C-49 , page 4
C-49 , page 1
C-49 , page 3
C-49 , page 6
C-49 , page 5
C-49 , page 9
C-49 , page 11
C-49 , page 14
C-49 , page 17
B-54 , page 96
B-54 , page 2
B-54 , page 4
U-1 , page 4
U-1 , page 1
U-1 , page 12
C-96 , page 97
C-96 , page 84
C-96 , page 96
C-96 , page 22
C-96 , page 65
C-96 , page 82
C-96 , page 20
C-96 , page 21
C-96 , page 69
C-96 , page 85
C-96 , page 95
B-89 , page

In [5]:
validation_test_split_task.populate_task_data(page_dict)

{'book_id': 'W-8',
 'book_title': 'CSMP Mathematics for the Upper Primary Grades Part 1',
 'page_id': 38,
 'page_image': 'azure-blob://adt-pipeline/evaluation/gold_standard/pages/W-8__page_38.png'}